# TP 8 — CNN 1D Classification de type de culture avec **PyTorch**

*Notebook miroir de TP 7 (Keras) — même cas d'usage, même dataset, API différente.*

## Introduction

Ce notebook reproduit exactement le meme cas d'usage que TP 7 (CNN 1D Keras),
en utilisant **PyTorch**. Il illustre les differences concretes entre les deux frameworks
pour les reseaux convolutionnels 1D.

**Difference critique Keras vs PyTorch pour Conv1D :**
- **Keras** : `Conv1D` attend `(batch, steps, channels)` — les channels sont la **derniere** dimension
- **PyTorch** : `nn.Conv1d` attend `(batch, channels, length)` — les channels sont **avant** la longueur
  => il faut transposer les donnees avec `.permute(0, 2, 1)` ou `.transpose(1, 2)`

**Objectifs pedagogiques :**
- Comprendre `nn.Conv1d`, `nn.MaxPool1d`, `nn.AdaptiveAvgPool1d`
- Ecrire une boucle d'entrainement complete avec validation
- Gerer la transposition des dimensions (batch, channels, length)
- Comparer avec TP 7 (meme architecture en Keras)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', DEVICE)

## 1. Chargement et preparation des donnees (identique a TP 7)

In [ ]:
df = pd.read_excel('Data/crop_csv_file.xlsx')
df['Season'] = df['Season'].str.strip()

FEATURES = ['Temperature', 'humidity', 'soil moisture', ' area', 'Crop_Year', 'Production']
TARGET   = 'Season'

df_clean = df[FEATURES + [TARGET]].dropna()
print('Shape:', df_clean.shape)
print('Distribution:', df_clean[TARGET].value_counts().to_dict())

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(df_clean[TARGET])
NUM_CLASSES = len(le.classes_)
print('Classes:', le.classes_)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURES]).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 2. Conversion en tenseurs PyTorch

**Difference cle avec Keras :**
- Keras `Conv1D` attend `(batch, steps, channels)` = `(n, 6, 1)`
- PyTorch `nn.Conv1d` attend `(batch, channels, length)` = `(n, 1, 6)`

On utilise `.permute(0, 2, 1)` pour transposer les dimensions.

In [ ]:
N_STEPS = X_train.shape[1]  # 6 features

# Conversion en tenseurs
# Keras : (n, steps, 1)  ->  PyTorch : (n, 1, steps)
X_train_t = torch.from_numpy(X_train).unsqueeze(1).to(DEVICE)  # (n, 1, 6)
X_test_t  = torch.from_numpy(X_test).unsqueeze(1).to(DEVICE)   # (n, 1, 6)
y_train_t = torch.from_numpy(y_train.astype(np.long)).to(DEVICE)
y_test_t  = torch.from_numpy(y_test.astype(np.long)).to(DEVICE)

print('X_train_t shape:', X_train_t.shape)  # (n, 1, 6) -- channels avant length
print('Comparer avec Keras : (n, 6, 1) -- channels apres steps')

# DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=64)

## 3. Architecture CNN 1D (PyTorch)

Architecture equivalente a TP 7 :
```
Input (channels=1, length=6)
  |
  v
Conv1d(in=1, out=32, kernel=2, padding=1)
  |
BatchNorm1d(32) + ReLU
  |
Conv1d(in=32, out=64, kernel=2, padding=1)
  |
AdaptiveAvgPool1d(1)   <- equivalent de GlobalAveragePooling1D()
  |                       reduit (batch, 64, L) en (batch, 64, 1)
Dropout(0.3)
  |
Linear(64, 64) -> ReLU -> Linear(64, NUM_CLASSES)
```

Note : `nn.CrossEntropyLoss` integre le softmax — pas besoin de couche Softmax explicite
(contrairement a Keras ou on specifie `activation='softmax'`).

In [ ]:
class CNN1DClassifier(nn.Module):
    """CNN 1D pour classification de type de culture.
    
    Equivalent PyTorch du modele Keras defini dans TP 7.
    Difference principale : l'entree est (batch, channels, length)
    alors que Keras attend (batch, steps, channels).
    """
    def __init__(self, n_classes, n_features=6):
        super(CNN1DClassifier, self).__init__()
        # Bloc convolutionnel 1 : equivalent Conv1D(32) + BatchNorm
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=2, padding=1)
        self.bn1   = nn.BatchNorm1d(32)
        # Bloc convolutionnel 2 : equivalent Conv1D(64)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=2, padding=1)
        # Pooling global : equivalent GlobalAveragePooling1D()
        self.gap   = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        # Couches fully-connected
        self.fc1   = nn.Linear(64, 64)
        self.fc2   = nn.Linear(64, n_classes)
        self.relu  = nn.ReLU()

    def forward(self, x):
        # x : (batch, 1, 6)
        x = self.relu(self.bn1(self.conv1(x)))  # (batch, 32, L)
        x = self.relu(self.conv2(x))             # (batch, 64, L)
        x = self.gap(x).squeeze(-1)              # (batch, 64)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))               # (batch, 64)
        return self.fc2(x)                       # (batch, n_classes) -- pas de softmax !


model = CNN1DClassifier(n_classes=NUM_CLASSES).to(DEVICE)
print(model)
print('\nNombre de parametres:',
      sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Boucle d'entrainement PyTorch

Note : `nn.CrossEntropyLoss` combine **LogSoftmax + NLLLoss** — pas besoin de softmax
dans le forward, ni de one-hot encoding de la cible (contrairement a Keras).

In [ ]:
# CrossEntropyLoss integre softmax — equivalent de 'categorical_crossentropy' Keras
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS    = 80
PATIENCE  = 10
best_val  = float('inf')
no_improve = 0
best_weights = None

train_losses, val_losses = [], []
train_accs, val_accs     = [], []

# Split validation (20% des donnees train)
val_size = int(len(train_dataset) * 0.2)
tr_size  = len(train_dataset) - val_size
tr_data, val_data = torch.utils.data.random_split(train_dataset, [tr_size, val_size])
tr_loader  = DataLoader(tr_data,  batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)

for epoch in range(EPOCHS):
    # ---- Entrainement ----
    model.train()
    correct, total, epoch_loss = 0, 0, 0.0
    for X_b, y_b in tr_loader:
        optimizer.zero_grad()
        out  = model(X_b)
        loss = criterion(out, y_b)  # CrossEntropy accepte des indices entiers
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_b)
        correct += (out.argmax(1) == y_b).sum().item()
        total += len(X_b)
    train_losses.append(epoch_loss / total)
    train_accs.append(correct / total)

    # ---- Validation ----
    model.eval()
    v_correct, v_total, v_loss = 0, 0, 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            out = model(X_b)
            v_loss += criterion(out, y_b).item() * len(X_b)
            v_correct += (out.argmax(1) == y_b).sum().item()
            v_total += len(X_b)
    val_losses.append(v_loss / v_total)
    val_accs.append(v_correct / v_total)

    # ---- Early stopping ----
    if val_losses[-1] < best_val:
        best_val = val_losses[-1]
        no_improve = 0
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print('Early stopping a l\'epoch', epoch + 1)
            break

    if (epoch + 1) % 10 == 0:
        print('Epoch {:3d} | Train Loss: {:.4f} Acc: {:.2%} | Val Loss: {:.4f} Acc: {:.2%}'.format(
              epoch+1, train_losses[-1], train_accs[-1], val_losses[-1], val_accs[-1]))

model.load_state_dict(best_weights)
print('Meilleur modele restaure (val_loss:', round(best_val, 4), ')')

In [ ]:
# Courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='Train Loss', color='#e67e22')
axes[0].plot(val_losses, label='Val Loss', color='#3498db')
axes[0].set_title('Loss — CNN 1D PyTorch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(train_accs, label='Train Acc', color='#e67e22')
axes[1].plot(val_accs, label='Val Acc', color='#3498db')
axes[1].set_title('Accuracy — CNN 1D PyTorch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('Epochs effectuees:', len(train_losses))

## 5. Evaluation

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        preds = model(X_b).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_b.cpu().numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

acc = (all_preds == all_true).mean()
print('Test Accuracy: {:.2%}'.format(acc))
print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=le.classes_))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matrice de confusion — CNN 1D PyTorch', fontsize=14)
plt.ylabel('Reel'); plt.xlabel('Predit')
plt.tight_layout(); plt.show()

## 6. Resume comparatif PyTorch vs Keras

| Aspect | PyTorch (ce notebook) | Keras (TP 7) |
|---|---|---|
| **Format entree** | `(batch, channels, length)` = `(n, 1, 6)` | `(batch, steps, channels)` = `(n, 6, 1)` |
| **Transposition** | `.unsqueeze(1)` pour ajouter la dimension channel | Reshape en `(n, 6, 1)` |
| **Convolution** | `nn.Conv1d(1, 32, kernel_size=2, padding=1)` | `Conv1D(32, kernel_size=2, padding='same')` |
| **Pooling global** | `nn.AdaptiveAvgPool1d(1)` + `.squeeze(-1)` | `GlobalAveragePooling1D()` |
| **Batch norm** | `nn.BatchNorm1d(32)` — prend nb de channels | `BatchNormalization()` — infere auto |
| **Loss** | `nn.CrossEntropyLoss` (integre softmax, accepte indices) | `categorical_crossentropy` (one-hot requis) |
| **Cible** | Indices entiers `[0, 1, 2, ...]` | One-hot `[[1,0,0], [0,1,0], ...]` |